### 0 Helper function

In [1]:
import yaml
import json

In [2]:
class helperfunction():
    def load_file(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return f.read()
    def load_yaml(self,filepath):
        with open(filepath,'r',encoding="utf-8") as f:
            return yaml.safe_load(f)
    def fop(self,float_num):
        return float(f"{float_num:.1f}")
    def jsonstr(self,ip):
        return str(json.dumps(ip,indent=4, ensure_ascii=False))

<hr>

### 01 Input CVResume

In [3]:
hp = helperfunction()
resume_json = hp.load_file("resume_json.txt")
print(resume_json)

{
  "contact_information": {
    "name": "Surya Teja Menta",
    "email": "-",
    "phone": "+91 8309584461",
    "linkedin": "-",
    "jobdb_link": "-",
    "portfolio_link": "suryatejamenta.co.in"
  },
  "professional_summary": {
    "has_summary": "Yes",
    "summary_points": [
      "I’m Surya Teja Menta, Results-driven Senior Data Scientist with 4+ years of experience in Data Science, Machine Learning (ML), and Generative AI (GenAI).",
      "Proven expertise in RAG (Retrieval-Augmented Generation), LLM fine-tuning, MLOps, and end-to-end AI solutions.",
      "Strong background in data analytics, statistical modeling, AI-powered automation, and scalable AI architectures.",
      "IBM Certified Professional Data Scientist with hands-on experience in LangChain, Hugging Face, OpenAI APIs, Vector Databases (ChromaDB, Pinecone), and cloud deployments (AWS, GCP).",
      "Passionate about AI research, model optimization, and developing cutting-edge AI solutions."
    ]
  },
  "education

<hr>

### 02 Promptbuilder

In [5]:
from google import genai
import json
import os
import yaml

In [9]:
class PromptBuilder(helperfunction):
    def __init__(self,section,criteria,targetrole,cvresume):
        self.section    = section
        self.criteria   = criteria[::-1]
        self.cvresume   = cvresume
        self.targetrole = targetrole
        self.config = self.load_yaml("prompt1.yaml")
        # print(self.config)
    def build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            }
        }
    def build(self):
        config_role      = self.config['role']['role1']
        config_objective = self.config['objective']['objective1']
        config_section   = self.config['section']['section1']
        config_expected  = self.config['expected_content'][self.section]
        config_criteria  = ""
        for item in self.criteria:
            config_criteria = f"- {item}\n" + config_criteria
        config_scale     = self.config['scale']['score1']
        # config_output    = self.config['output']['format1']
        
        prompt_role      = f"Role :\n{config_role}"+"\n\n"
        prompt_objective = f"objectvie :\n{config_objective}"+"\n"
        prompt_section   = f"section :\n{config_section}"+"\n\n"
        prompt_expected  = f"expected :\n{config_expected}"+"\n"
        prompt_criteria  = f"Criteria :\n{config_criteria}"+"\n"
        prompt_scale     = f"Scale :\n{config_scale}"+"\n"
        prompt_output    = f"output :\n{json.dumps(self.build_response_template(), indent=2)}"+"\n\n"
        prompt_cvresume  = f"CV/Resume: \n{self.cvresume}"+"\n"

        prompt = prompt_role + prompt_objective + prompt_section \
            + prompt_expected + prompt_criteria + prompt_scale \
            + prompt_output + prompt_cvresume
        prompt = prompt.replace("<section_name>",self.section)
        prompt = prompt.replace("<targetrole>",self.targetrole)
        return prompt

In [19]:
p1 = PromptBuilder( 
    section    = "Profile", 
    criteria   = ["Completeness", "ContentQuality"],
    targetrole = "Data science",
    cvresume   = resume_json
)
prompt1 = p1.build()
print(prompt1)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Summary section from the resume using the scoring criteria.
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale below.

section :
You are evaluating the Summary section.

expected :
Content checklist:
- 2-4 sentence summary of experience
- Technical & domain strengths
- Career focus & value proposition
- Avoid buzzwords and vague claims
Feedback guideline:
- Feedback should be ~20 words.
- Highlight clarity, focus, and alignment with Data science.
Few-shot examples (Summary quality):
Score 5 (excellent):
"Senior data scientist with 5+ years building ML models (classification, forecasting, recommendation) using Python and SQL, leading end-to-end projects that improve product and business KPIs."
Score 3 (sufficient):
"Data professional with experience in data analysis and basic machine learning, intere

In [18]:
p2 = PromptBuilder( 
    section  = "Summary", 
    criteria = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt2 = p2.build()
print(prompt2)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Summary section from the resume using the scoring criteria.
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale below.


section :
You are evaluating the Summary section.

expected :
Content checklist:
- 2–4 sentence summary of experience
- Technical & domain strengths
- Career focus & value proposition
- Avoid buzzwords and vague claims

Feedback guideline:
- Feedback should be ~20 words.
- Highlight clarity, focus, and alignment with Data science.

Few-shot examples (Summary quality):

Score 5 (excellent):
"Senior data scientist with 5+ years building ML models (classification, forecasting, recommendation) using Python and SQL, leading end-to-end projects that improve product and business KPIs."

Score 3 (sufficient):
"Data professional with experience in data analysis and basic machine learning, i

In [21]:
p3 = PromptBuilder( 
    section  = "Education", 
    criteria = ["Completeness","RoleRelevance"],
    targetrole = "Data science",
    cvresume = resume_json
)
prompt3 = p3.build()
print(prompt3)

Role :
You are the expert HR evaluator

objectvie :
Evaluate the Education section from the resume using the scoring criteria.
Measure how well the candidate matches the Data science role.
Consider: degree relevance, experience alignment, skills/tools, and seniority evidence.
Score 0-5 using the scale below.


section :
You are evaluating the Education section.

expected :
Content checklist:
- Institution name
- Degree & field of study
- Dates attended
- GPA, honors (optional)
- Relevance to data/tech career (if applicable)

Feedback guideline:
- Feedback should be ~20 words.
- Comment on clarity, relevance, and any missing key information.

Few-shot examples (Education quality):

Score 5 (excellent):
"B.Sc. in Computer Engineering, Chulalongkorn University (2017–2021), GPA 3.6/4.0, coursework in ML, statistics, and data mining."

Score 3 (sufficient):
"B.Eng. in Electrical Engineering, major in control systems, graduated 2020."

Score 1 (poor):
"Bachelor degree, graduated a few years 